# Compile a model to `.hef` for Hailo-8L (smoke test: YOLOv8n COCO)

AI_PLAN T0.2 / T0.5, decision D-005. Runs on Google Colab (Linux x86_64), never on the robot.

**Before running**
1. Put `hailo_dataflow_compiler-<ver>-py3-none-linux_x86_64.whl` in Google Drive (default `MyDrive/hailo/`).
2. Set `MZ_TAG` below: open https://github.com/hailo-ai/hailo_model_zoo/releases and pick the release whose notes name the same Dataflow Compiler version as your wheel.
3. Optional: *Runtime → Change runtime type → T4 GPU* (higher optimization level; CPU works but is slower and quantizes worse).

Output: `.hef` + compile log in `MyDrive/hailo/out/`, file name carries the DFC version (needed later to match HailoRT on the Pi).

In [ ]:
# --- Parameters -------------------------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/hailo"
DFC_WHEEL = "hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl"
MZ_TAG = ""            # e.g. "v2.xx" — must match the DFC version (see release notes)
MODEL = "yolov8n"      # Model Zoo network name
HW_ARCH = "hailo8l"
PYTHON = "3.10"        # DFC wheel requirement (also the AI code version, D-017)

assert MZ_TAG, "Set MZ_TAG to the Model Zoo release matching your DFC version"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
WHEEL_PATH = os.path.join(DRIVE_DIR, DFC_WHEEL)
assert os.path.isfile(WHEEL_PATH), f"DFC wheel not found: {WHEEL_PATH}"

## Install: Python 3.10 venv (Colab's default Python is newer), DFC, Model Zoo

In [ ]:
%%bash -s "$WHEEL_PATH" "$MZ_TAG" "$PYTHON"
set -euo pipefail
apt-get -qq update && apt-get -qq install -y graphviz libgraphviz-dev > /dev/null
pip -q install uv
uv venv --python "$3" /content/venv
uv pip install --python /content/venv/bin/python "$1"
rm -rf /content/hailo_model_zoo
git clone -q --depth 1 --branch "$2" https://github.com/hailo-ai/hailo_model_zoo.git /content/hailo_model_zoo
uv pip install --python /content/venv/bin/python -e /content/hailo_model_zoo

In [ ]:
%%bash
set -euo pipefail
export PATH=/content/venv/bin:$PATH
python -c "from importlib.metadata import version; print('DFC', version('hailo-dataflow-compiler'))"
hailomz --help > /dev/null && echo "hailomz OK"

## Calibration images

Smoke test only: 128 COCO images. For real robot models use 500–1000 frames from the robot camera (AI_PROPOSAL §9.2).

In [ ]:
%%bash
set -euo pipefail
cd /content
[ -d coco128 ] || { wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip && unzip -q coco128.zip; }
ls coco128/images/train2017 | wc -l

## Compile

Without `--ckpt`, `hailomz` downloads the Model Zoo's pretrained ONNX for `MODEL`. For our own models pass `--ckpt <file>.onnx` (and `--yaml` if the head differs).

In [ ]:
%%bash -s "$MODEL" "$HW_ARCH"
set -euo pipefail
export PATH=/content/venv/bin:$PATH
export USER="${USER:-root}"  # DFC compile reads $USER; Colab does not set it
mkdir -p /content/out && cd /content/out
hailomz compile "$1" --hw-arch "$2" --calib-path /content/coco128/images/train2017 2>&1 | tee compile.log
ls -la *.hef

In [ ]:
import glob, hashlib, shutil, subprocess
# hailo_sdk_client.__version__ is empty; read the installed wheel version instead
dfc = subprocess.run(["/content/venv/bin/python", "-c",
                      "from importlib.metadata import version; print(version('hailo-dataflow-compiler'))"],
                     capture_output=True, text=True, check=True).stdout.strip()
assert dfc, "Could not read DFC version"
out_dir = os.path.join(DRIVE_DIR, "out")
os.makedirs(out_dir, exist_ok=True)
for hef in glob.glob("/content/out/*.hef"):
    name = f"{MODEL}_{HW_ARCH}_dfc{dfc}.hef"
    shutil.copy(hef, os.path.join(out_dir, name))
    sha = hashlib.sha256(open(hef, "rb").read()).hexdigest()
    print(name, sha)  # record in models/model_registry.yaml
shutil.copy("/content/out/compile.log", os.path.join(out_dir, f"{MODEL}_{HW_ARCH}_dfc{dfc}.log"))